# CUAD — Tokenization handoff (BERT baseline)

**Input:** final annotation-audited training chunks from `01_CUAD_final_exploration_and_chunking.ipynb`. **Output:** tokenized short chunks, overlength review file, category mapping, and a summary. This notebook **does not train a model**.

**Important:** BERT is a temporary baseline. Confirm the model with teammates; its tokenizer must match the model. This notebook never silently truncates long contract chunks. It excludes overlength chunks from the immediately trainable subset and preserves them for later resolution. **Do not interpret this subset as retaining 100% annotated-answer coverage.**

## 1. Setup
Run `pip install transformers` in your environment if necessary. The first tokenizer load may require internet access.

In [3]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Uncomment and run only if transformers is not installed:
# %pip install -U transformers
import json
from pathlib import Path
from collections import Counter
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"  # Change only after agreeing on a base model.
MAX_LENGTH = 512
INPUT_DIR = Path("../data/processed")
OUTPUT_DIR = INPUT_DIR / "tokenized_bert"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

chunks = json.loads((INPUT_DIR / "final_training_chunks.json").read_text(encoding="utf-8"))
categories = json.loads((INPUT_DIR / "cuad_categories.json").read_text(encoding="utf-8"))
assert chunks and categories
assert len(set(categories)) == len(categories)
assert all(isinstance(c["text"], str) and isinstance(c["labels"], list) for c in chunks)
assert all(set(c["labels"]) <= set(categories) for c in chunks)
print("Loaded chunks:", len(chunks), "Categories:", len(categories))

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loaded chunks: 13003 Categories: 41


## 2. Load matching tokenizer and inspect token lengths
Token counts include special tokens; word counts are not sufficient.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model_limit = tokenizer.model_max_length
assert model_limit < 100000 and MAX_LENGTH <= model_limit, f"Check model limit: {model_limit}"
lengths = [len(tokenizer(c["text"], add_special_tokens=True, truncation=False)["input_ids"]) for c in chunks]
ordered = sorted(lengths)
def percentile(p):
    return ordered[min(len(ordered)-1, int((len(ordered)-1)*p))]
print("Tokenizer:", MODEL_NAME, "Model max:", model_limit)
print("Min / median / p90 / p95 / p99 / max:", min(lengths), *(percentile(p) for p in (0.5,0.9,0.95,0.99)), max(lengths))
print("Over length:", sum(x > MAX_LENGTH for x in lengths), "of", len(lengths))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (538 > 512). Running this sequence through the model will result in indexing errors


Tokenizer: bert-base-uncased Model max: 512
Min / median / p90 / p95 / p99 / max: 62 402 497 565 887 5515
Over length: 1103 of 13003


## 3. Encode multi-label targets and split by sequence length

Each chunk can have several categories. We encode a **multi-hot vector**, not a single class ID. We **do not pad to 512** here; teammates can use a dynamic-padding collator. Chunks longer than 512 tokens are saved separately rather than truncated, because truncation could remove the actual clause evidence.

In [6]:
label2id = {label: i for i, label in enumerate(categories)}
id2label = {str(i): label for label, i in label2id.items()}
ready, overlength = [], []
for c, token_len in zip(chunks, lengths):
    labels = [0.0] * len(categories)
    for label in c["labels"]:
        labels[label2id[label]] = 1.0
    metadata = {"contract_title": c["contract_title"], "chunk_id": c["chunk_id"],
                "char_start": c["char_start"], "char_end": c["char_end"],
                "category_names": c["labels"], "token_length": token_len}
    if token_len > MAX_LENGTH:
        overlength.append({**metadata, "text": c["text"], "labels": labels})
    else:
        encoded = tokenizer(c["text"], truncation=False, padding=False,
                            add_special_tokens=True, return_attention_mask=True)
        assert len(encoded["input_ids"]) == token_len
        ready.append({**metadata, "input_ids": encoded["input_ids"],
                      "attention_mask": encoded["attention_mask"], "labels": labels})
print("Ready for BERT:", len(ready))
print("Overlength (needs model/strategy decision):", len(overlength))
print("Ready positive / negative:", sum(any(x["labels"]) for x in ready),
      sum(not any(x["labels"]) for x in ready))
assert len(ready) + len(overlength) == len(chunks)

Ready for BERT: 11900
Overlength (needs model/strategy decision): 1103
Ready positive / negative: 4742 7158


## 4. Save teammate handoff files

- `bert_ready.jsonl`: variable-length input IDs, attention masks, multi-hot labels, and contract identifiers.
- `overlength_review.jsonl`: chunks requiring a long-context model or answer-safe re-chunking.
- `label_mapping.json`: stable class order.
- `tokenization_summary.json`: counts and settings.

**Before model training:** create train/validation splits **by contract**, never by chunk, to prevent overlap leakage. Keep the original CUAD test contracts separate. Use `BCEWithLogitsLoss` for multi-label targets, not softmax cross-entropy. The 512-token subset is not a complete representation of all annotated answers.

In [7]:
def save_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

save_jsonl(OUTPUT_DIR / "bert_ready.jsonl", ready)
save_jsonl(OUTPUT_DIR / "overlength_review.jsonl", overlength)
(OUTPUT_DIR / "label_mapping.json").write_text(json.dumps({"label2id":label2id, "id2label":id2label}, indent=2), encoding="utf-8")
summary = {"model_name": MODEL_NAME, "max_length": MAX_LENGTH,
           "total_chunks": len(chunks), "ready_chunks": len(ready),
           "overlength_chunks": len(overlength), "category_count": len(categories),
           "max_observed_tokens": max(lengths),
           "ready_positive": sum(any(r["labels"]) for r in ready),
           "ready_negative": sum(not any(r["labels"]) for r in ready),
           "note": "Overlength chunks excluded, not truncated. Contract-level split required."}
(OUTPUT_DIR / "tokenization_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("Saved in:", OUTPUT_DIR.resolve())

{
  "model_name": "bert-base-uncased",
  "max_length": 512,
  "total_chunks": 13003,
  "ready_chunks": 11900,
  "overlength_chunks": 1103,
  "category_count": 41,
  "max_observed_tokens": 5515,
  "ready_positive": 4742,
  "ready_negative": 7158,
  "note": "Overlength chunks excluded, not truncated. Contract-level split required."
}
Saved in: /Users/rashipatel/Documents/Accenture Project /Accenture-1G-contract-review-challenge/data/processed/tokenized_bert


## Conclusion

Tokenization converts each chunk into the base model's token IDs and attention mask. We preserve **multiple clause labels** per chunk. The short subset is ready for a BERT-style multi-label baseline after a contract-level validation split. **Overlength chunks are not discarded from the project:** they remain in a separate file. Teammates should choose either a long-context model or an annotation-safe, tokenizer-aware chunking strategy before claiming complete answer coverage. Because the input chunks were generated using training annotations, the same boundary-selection method cannot be used on unlabeled inference documents; design an inference-safe method separately.

### Next Step: Building the Base Model

We have finished preparing and tokenizing our training data. The next step is to use this data to build our base model. Our goal is to train a model that can identify different types of clauses in contracts. Since one chunk can contain more than one type of clause, we are treating this as a multi-label classification problem.

### What We Have So Far

- **Total chunks:** 13,003
- **Total categories:** 41
- **Tokenizer:** bert-base-uncased
- **Maximum token length:** 512
- **Chunks within the limit:** 11,900
- **Chunks over the limit:** 1,103

### What We Need to Do Next

1. **Load the tokenized data:** We can use the files generated during tokenization. These include token IDs, attention masks, and labels for each chunk.

2. **Choose our base model:** Since we used the BERT tokenizer, we can start with **bert-base-uncased**. If we decide to use another model, we may need to update our tokenizer.

3. **Prepare training and validation data:** We need to make sure that chunks from the same contract do not appear in different datasets. This will help us avoid data leakage.

4. **Handle longer chunks:** We found 1,103 chunks that are longer than 512 tokens. We need to decide how to handle them because simply cutting the text could remove important information.

5. **Train the model:** We can start with a simple training setup and check whether the model is learning to identify the different clause categories. Since chunks can have multiple labels, we need to use a multi-label classification setup.

6. **Evaluate the model:** After training, we should check precision, recall, and F1-score to understand how well our model is performing. We can also look at which categories are harder for the model to identify.

7. **Save our results:** We should save the trained model, tokenizer, category mapping, and evaluation results so we can use them later or compare them with other models.

### NOTES

Our current tokenized data comes from the training contracts. We still need to prepare validation and test data using the same process.

Also, we should not ignore the longer chunks without checking them, because some may contain important annotated clauses.

### Our Goal

For now, we want to build a working base model and see how well it performs on our data. Once we have the initial results, we can understand what is working, what needs improvement, and what changes we should try next.